# Cold-Start Anomaly Detection for Concrete Surface Crack Inspection using PatchCore

**Author:** Lam Huynh Khang — SE200666 — FPT University HCMC  
**Supervisor:** Nguyen Xuan Huy, Lecturer, FPT University  

This notebook implements the complete baseline and experimental pipeline for the project proposal **Cold-Start Anomaly Detection for Concrete Surface Crack Inspection using PatchCore with Pixel-Level Heatmap Localization**. It covers the evaluation of unsupervised PatchCore against supervised CNN classifiers.

### Objectives & Research Questions:
- **RQ1 (Detection reliability):** Detect cracks reliably with 0 defect images during training using PatchCore.
- **RQ2 (Pixel localization):** Localize crack regions via anomaly heatmaps compared against pseudo-ground-truth masks.
- **RQ3 (Coreset ablation):** Test how coreset subsampling ratio (1%, 5%, 10%, 25%, 50%, 100%) impacts AUROC and speed.
- **RQ4 (Data efficiency):** Compare PatchCore (0 training defects) against supervised classifiers (5, 10, 50, 100, 200 training defects).

## 1. Setup & Dependencies
Clone the repository, install required libraries, and install the `patchcore` package in editable mode.

In [ ]:
# Remove old clone if any, clone latest version from repo
!rm -rf PatchCore-SurfaceCrackInspection
!git clone https://github.com/Welkie/PatchCore-SurfaceCrackInspection.git
%cd PatchCore-SurfaceCrackInspection

# Install required libraries and patchcore package in editable mode
!pip install timm faiss-gpu scikit-image scikit-learn matplotlib tqdm pillow click pandas seaborn opencv-python-headless
!pip install -e .

## 2. Verify Environment & Imports
Add `src/` and `bin/` to the Python path and verify that the custom dataset class can be imported.

In [ ]:
import os
import sys

# Add src/ and bin/ to python path
sys.path.append(os.path.abspath('src'))
sys.path.append(os.path.abspath('bin'))

print("Current workspace:", os.getcwd())
print("Files in workspace:", os.listdir('.'))

# Verify dataset class imports successfully
from patchcore.datasets.concrete import ConcreteDataset, DatasetSplit
print("\nDataset class successfully imported and registered!")

## 3. Verify Kaggle Dataset Path
The Kaggle Surface Crack Detection dataset must be added to the notebook. We verify that the dataset path is accessible and check for the `Negative` and `Positive` folders.

In [ ]:
import os

# Try multiple known Kaggle dataset path patterns
possible_paths = [
    "/kaggle/input/datasets/arunrk7/surface-crack-detection",
    "/kaggle/input/surface-crack-detection",
    "/kaggle/input/surface-crack-detection/surface-crack-detection",
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise ValueError(
        "Dataset path not found. Please add 'arunrk7/surface-crack-detection' "
        "to your Kaggle Notebook input."
    )
else:
    print(f"Successfully located dataset at: {data_path}")
    print("Folders in dataset:", os.listdir(data_path))

## 4. Execute End-to-End Experiments (Ablation & Baselines)
We run `bin/run_experiments.py` which automates:
1. Fitting and evaluating PatchCore on WideResNet-50 and ResNet-18 for all coreset ratios (1%, 5%, 10%, 25%, 50%, 100%).
2. Training and evaluating supervised classifiers (ResNet-50, EfficientNet-B0) on varying positive training samples (5, 10, 50, 100, 200).
3. Pre-generating comparative plots and saving all results under the `results/` folder.

In [ ]:
# Run end-to-end experiment pipeline
!python bin/run_experiments.py --data_path {data_path} --results_path results --gpu 0

## 5. View Quantitative Results
Load the results tables into Pandas DataFrames to analyze the performance of PatchCore and the supervised baselines.

In [ ]:
import pandas as pd
import os
from IPython.display import display

print("=" * 60)
print("TABLE 1: PATCHCORE CORESET ABLATION RESULTS (RQ3)")
print("=" * 60)
pc_path = "results/patchcore_ablation_results.csv"
if os.path.exists(pc_path):
    df_pc = pd.read_csv(pc_path)
    display(df_pc)
else:
    print("PatchCore results not found. Experiment may not have completed yet.")

print()
print("=" * 60)
print("TABLE 2: SUPERVISED BASELINE DATA EFFICIENCY RESULTS (RQ4)")
print("=" * 60)
sup_path = "results/supervised/results.csv"
if os.path.exists(sup_path):
    df_sup = pd.read_csv(sup_path)
    if len(df_sup) > 0:
        display(df_sup)
    else:
        print("Supervised results CSV exists but is empty. Experiments may still be running.")
else:
    print("Supervised results not found. Experiments may still be running.")

## 6. Scientific Visualization Plots
Render the generated evaluation plots directly in the notebook to visually answer RQ3 and RQ4.

In [ ]:
from PIL import Image
import os
import matplotlib.pyplot as plt

plots = [
    ("results/plot_auroc_vs_coreset.png", "Coreset Subsampling Impact on Image AUROC"),
    ("results/plot_latency_vs_coreset.png", "Inference Speed vs. Coreset Subsampling Ratio"),
    ("results/plot_data_efficiency.png", "Data Efficiency Curve: Unsupervised PatchCore vs. Supervised CNNs"),
]

found_any = False
for p, title in plots:
    if os.path.exists(p):
        found_any = True
        print(f"\n{'='*60}")
        print(f"{title}")
        print(f"{'='*60}")
        img = Image.open(p)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"\n[SKIP] Plot not found: {p}")

if not found_any:
    print("\nNo plots were found. Please re-run Cell 4 (experiments) first.")

## 7. Qualitative Visual Heatmaps (RQ2)
Display the qualitative visual localization. For each tested backbone, this shows the Input Image, the generated Pseudo-Ground-Truth Mask, and the resulting PatchCore Anomaly Heatmap.

In [ ]:
from PIL import Image
import os
import matplotlib.pyplot as plt

found_any = False
for backbone in ["resnet18", "resnet50"]:
    p = f"results/visual_localization_{backbone}.png"
    if os.path.exists(p):
        found_any = True
        print(f"\n{'='*60}")
        print(f"Qualitative Crack Heatmap Overlay ({backbone})")
        print(f"{'='*60}")
        img = Image.open(p)
        plt.figure(figsize=(12, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"\n[SKIP] Heatmap not found for {backbone}: {p}")

if not found_any:
    print("\nNo heatmap visualizations were generated. Please re-run Cell 4 (experiments) first.")